# Project Setup — the MCP CLI chatbot

The rest of this section builds a **CLI chatbot** that talks to documents through MCP. The code lives in **standalone project folders** (not notebook cells). To keep the repo usable as a hands-on guide, there are **two copies**:

| Folder | Role |
|--------|------|
| [`cli-project/`](./cli-project) | **Starter** — `TODO`s intact. Do the exercises here. Committed once, pristine. |
| [`cli-project-complete/`](./cli-project-complete) | **Solution / answer key** — every lesson done. Runs out of the box; peek if stuck. |

Each lesson's **notebook** is the guide (what to change, with the code shown). The `.py` files are the source of truth; the notebooks explain and verify them.

## Keeping the starter pristine

So the starter stays a real exercise (no answers baked in), two things protect it:

1. **Future lesson commits touch notebooks only** — both project folders are committed once, then frozen. Nothing overwrites the starter unless a `cli-project/` file is explicitly staged (we don't).
2. **`skip-worktree` guard** — `cli-project/mcp_server.py` and `cli-project/mcp_client.py` are flagged `skip-worktree`, so your local edits to them are **invisible to git** (won't show in `status`, can't be staged). Edit and run them freely to practice; the committed pristine version stays safe.

To view or lift the flag later:

```bash
git ls-files -v building-with-the-claude-api/07-mcp/cli-project/ | grep '^S'   # S = skip-worktree
git update-index --no-skip-worktree <file>                                     # stop ignoring edits
```

(The flag is local to this clone — a fresh clone gets the pristine starter and can set the same.)

## Project layout

```
cli-project/  (and identical cli-project-complete/)
├── main.py            # entrypoint: wires client + server + chat loop
├── mcp_server.py      # the MCP SERVER (tools / resources / prompts)  <- exercises edit this
├── mcp_client.py      # the MCP CLIENT (list/call tools, prompts, resources)  <- exercises edit this
├── core/
│   ├── claude.py      # Anthropic wrapper (chat, message helpers)
│   ├── chat.py        # base tool-use loop
│   ├── cli_chat.py    # CLI-specific chat (document mentions, commands)
│   ├── cli.py         # prompt-toolkit terminal UI
│   └── tools.py       # ToolManager: aggregates tools across MCP clients
├── pyproject.toml / uv.lock
└── .env               # config (gitignored — you create/populate this)
```

In the **starter**, `mcp_server.py` / `mcp_client.py` are full of `TODO`s. Lessons fill them in: **Defining tools**, **Implementing a client**, **Defining resources**, **Accessing resources**, **Defining prompts**, **Prompts in the client**. The **complete** folder already has all of it.

## Adaptations from the course

| Course default | This repo | Why |
|----------------|-----------|-----|
| `CLAUDE_MODEL="claude-sonnet-4-5"` | **`claude-haiku-4-5-20251001`** | `core/claude.py` always sends `temperature=1.0`, which **400s on flagships**; the bare `claude-sonnet-4-5` alias is risky. Haiku 4.5 supports temperature + tool use and matches the repo. |
| `USE_UV=1` (uv) | **`USE_UV=0`** (pip) | No `uv` on the dev machine. Pip mode runs the server via `python mcp_server.py`. |
| deps via `uv`/`uv.lock` | shared repo **`.venv`** | `mcp[cli]` + `prompt-toolkit` are in the repo `requirements.txt`. |

> Have `uv`? Set `USE_UV=1` and `uv run main.py` instead — it manages its own env from `pyproject.toml`.

## One-time setup for a fresh clone

1. `pip install -r requirements.txt` (from the repo root) into your venv.
2. Create a `.env` in **whichever folder you'll run** (`cli-project/` and/or `cli-project-complete/`) — they're gitignored, so a clone has none. Each needs:
   ```env
   CLAUDE_MODEL="claude-haiku-4-5-20251001"
   ANTHROPIC_API_KEY="sk-ant-..."
   USE_UV=0
   ```
   ...or run the **convenience cell** below to scaffold both from your repo-root `.env`.
3. Run the setup check.

## Convenience: scaffold both `.env` files from the repo-root `.env`

Copies `ANTHROPIC_API_KEY` from the repo-root `.env` into each project folder's `.env` (only if missing there) and scaffolds `CLAUDE_MODEL` + `USE_UV`. Idempotent; never prints the key; both `.env`s stay gitignored.

In [ ]:
import os
from pathlib import Path
from dotenv import find_dotenv

repo_root = Path(os.path.dirname(find_dotenv()))
root_env = repo_root / ".env"
mcp_dir = repo_root / "building-with-the-claude-api" / "07-mcp"
targets = [mcp_dir / "cli-project" / ".env", mcp_dir / "cli-project-complete" / ".env"]


def read_env(path):
    d = {}
    if path.exists():
        for line in path.read_text().splitlines():
            s = line.strip()
            if s and not s.startswith("#") and "=" in s:
                k, _, v = s.partition("=")
                d[k.strip()] = v.strip().strip('"')
    return d


root_key = read_env(root_env).get("ANTHROPIC_API_KEY", "")

for proj_env in targets:
    proj = read_env(proj_env)
    if not proj.get("ANTHROPIC_API_KEY") and root_key:
        proj["ANTHROPIC_API_KEY"] = root_key
    proj.setdefault("CLAUDE_MODEL", "claude-haiku-4-5-20251001")
    proj.setdefault("USE_UV", "0")

    model = proj["CLAUDE_MODEL"]
    key = proj.get("ANTHROPIC_API_KEY", "")
    use_uv = proj["USE_UV"]

    proj_env.parent.mkdir(parents=True, exist_ok=True)
    lines = [
        'CLAUDE_MODEL="' + model + '"',
        "ANTHROPIC_API_KEY=" + key,
        "",
        "# 1 if using uv, 0 for pip",
        "USE_UV=" + use_uv,
    ]
    proj_env.write_text("\n".join(lines) + "\n")

    status = "set" if key else "MISSING"
    print(proj_env.parent.name + "/.env  ->  key " + status + ", model " + model + ", USE_UV=" + use_uv)

## Setup check

Confirms the dependencies import and that each project `.env` has the required vars (reports **set**, never prints the value).

In [ ]:
import importlib
import os
from pathlib import Path
from dotenv import find_dotenv

print("Dependencies:")
for pkg in ["anthropic", "mcp", "prompt_toolkit", "dotenv"]:
    try:
        importlib.import_module(pkg)
        print("  OK       " + pkg)
    except ImportError:
        print("  MISSING  " + pkg + "  -> pip install -r requirements.txt")

mcp_dir = Path(os.path.dirname(find_dotenv())) / "building-with-the-claude-api" / "07-mcp"
for folder in ["cli-project", "cli-project-complete"]:
    proj_env = mcp_dir / folder / ".env"
    print("\n" + folder + "/.env:")
    if not proj_env.exists():
        print("  NOT FOUND - run the convenience cell above.")
        continue
    present = {}
    for line in proj_env.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            present[k.strip()] = bool(v.strip().strip('"'))
    for need in ["CLAUDE_MODEL", "ANTHROPIC_API_KEY", "USE_UV"]:
        print("  " + need + ": " + ("set" if present.get(need) else "MISSING or empty"))

## Running the app

Interactive terminal app — run it in a shell (not this notebook). Activate the venv first (pip mode spawns the server as `python mcp_server.py`, so `python` must be the venv's):

```powershell
# from the repo root
.\.venv\Scripts\Activate.ps1
cd building-with-the-claude-api\07-mcp\cli-project-complete
python main.py
```

Ask **"what's 1+1?"** — a quick reply confirms the wiring. Quit with Ctrl-C.

- **`cli-project-complete/`** works immediately (all lessons done).
- **`cli-project/`** (starter) launches and answers general questions, but can't touch documents until *you* fill in the `TODO`s per lesson.

Next: **Defining tools with MCP**.